# Step 2 — Final VisDrone Person-Only Dataset Audit (Google Colab)

This notebook is the **final Step-2 technical pipeline** for the aerial human-detection project.  
All code is in English and all outputs are saved to Google Drive.

It performs:

- official VisDrone2019-DET acquisition/discovery;
- raw inventory and provenance;
- annotation parsing and QC;
- `pedestrian (1) + people (2) -> person`;
- image integrity, brightness, blur and near-black checks;
- invalid/out-of-bounds box checks;
- exact and perceptual cross-split duplicate checks;
- tiny/small person statistics;
- train/validation distribution-shift tests;
- 16 graphical figures;
- annotated sample panels;
- canonical YOLO, COCO and VisDrone person-only exports;
- manifest, source registry and checksums;
- YOLO/COCO parity validation;
- Step-2 gate report;
- automated HTML/PDF technical reports;
- Drive ZIP delivery;
- **final cell for auditing the dataset exported by Aerial Person Studio GUI**.

> The official VisDrone test-dev split is only a benchmark reference.  
> The project's private final test is separate and must contain at least 1,000 independently collected/reviewed aerial images.

In [1]:
# CELL 1 — Install dependencies
import sys, subprocess, pkgutil
req = {
    "gdown":"gdown>=5.2.0", "cv2":"opencv-python-headless>=4.9",
    "PIL":"Pillow>=10", "imagehash":"ImageHash>=4.3.1",
    "pandas":"pandas>=2.1", "numpy":"numpy>=1.26",
    "matplotlib":"matplotlib>=3.8", "scipy":"scipy>=1.11",
    "yaml":"PyYAML>=6", "tqdm":"tqdm>=4.66", "reportlab":"reportlab>=4"
}
missing=[v for k,v in req.items() if pkgutil.find_loader(k) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
print("Dependencies ready.")

/tmp/ipykernel_989/725005482.py:10: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  missing=[v for k,v in req.items() if pkgutil.find_loader(k) is None]


Dependencies ready.


In [2]:
# CELL 2 — Mount Drive and configuration
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, json, random, shutil, zipfile, hashlib, math, base64
import numpy as np, pandas as pd
from collections import Counter
from tqdm.auto import tqdm

SEED=42
random.seed(SEED); np.random.seed(SEED)

DRIVE_ROOT=Path("/content/drive/MyDrive/AerialHumanDetection/Step2_VisDrone_Final")
ARCHIVE_DIR=DRIVE_ROOT/"01_source_archives"
REPORT_DIR=DRIVE_ROOT/"02_reports"
TABLE_DIR=REPORT_DIR/"tables"
FIG_DIR=REPORT_DIR/"figures"
SAMPLE_DIR=REPORT_DIR/"annotated_samples"
PROV_DIR=DRIVE_ROOT/"03_provenance"
DELIVERY_DIR=DRIVE_ROOT/"04_delivery"

LOCAL_ROOT=Path("/content/step2_visdrone")
RAW_ROOT=LOCAL_ROOT/"raw"
BUILD_ROOT=LOCAL_ROOT/"person_only_build"

for p in [ARCHIVE_DIR,TABLE_DIR,FIG_DIR,SAMPLE_DIR,PROV_DIR,DELIVERY_DIR,RAW_ROOT,BUILD_ROOT]:
    p.mkdir(parents=True,exist_ok=True)

AUTO_DOWNLOAD=True
INCLUDE_TEST_DEV=True
EXPORT_IMAGES=True
NEAR_DUP_HAMMING_THRESHOLD=4
NEAR_DUP_MAX_PAIRS=10000
PERSON_CATEGORIES={1:"pedestrian",2:"people"}
print("Drive output:",DRIVE_ROOT)

Mounted at /content/drive
Drive output: /content/drive/MyDrive/AerialHumanDetection/Step2_VisDrone_Final


In [6]:
# CELL 3 — Download/extract official VisDrone2019-DET
import gdown

OFFICIAL={
 "train":("VisDrone2019-DET-train.zip","1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn"),
 "val":("VisDrone2019-DET-val.zip","1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59"),
 "test-dev":("VisDrone2019-DET-test-dev.zip","1PFdW_VFSCfZ_sTSZAGjQdifF_Xd5mf0V"),
}
splits=["train","val"]+(["test-dev"] if INCLUDE_TEST_DEV else [])

def get_archive(split):
    name,gid=OFFICIAL[split]; dst=ARCHIVE_DIR/name
    if dst.exists() and dst.stat().st_size>1_000_000:
        print("[FOUND]",dst.name); return dst
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(dst)
    print("[DOWNLOAD]",split)
    gdown.download(f"https://drive.google.com/uc?id={gid}",str(dst),quiet=False,fuzzy=True)
    if not dst.exists(): raise RuntimeError(f"Download failed: {dst}")
    return dst

def extract(split,zp):
    root=RAW_ROOT/split; marker=root/".ok"
    if marker.exists(): return root
    if root.exists(): shutil.rmtree(root)
    root.mkdir(parents=True)
    with zipfile.ZipFile(zp) as z: z.extractall(root)
    marker.write_text("ok"); return root

archives={s:get_archive(s) for s in splits}
roots={s:extract(s,z) for s,z in archives.items()}
print("Archives/extraction ready.")

[DOWNLOAD] train


FileURLRetrievalError: Failed to retrieve file url:

	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. If the file you are trying to
	access is particularly large or is shared with many people, it may
	take up to 24 hours to be able to view or download the file. If you
	still can't access a file after 24 hours, contact your domain
	administrator.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn

but Gdown can't. Please check connections and permissions.

In [ ]:
# CELL 4 — Discover folders and define parser/utilities
import cv2, imagehash
from PIL import Image, ImageDraw

IMAGE_EXTS={".jpg",".jpeg",".png",".bmp"}
CATS={0:"ignored_regions",1:"pedestrian",2:"people",3:"bicycle",4:"car",5:"van",
      6:"truck",7:"tricycle",8:"awning-tricycle",9:"bus",10:"motor",11:"others"}

def find_dirs(root):
    imgs=[p for p in root.rglob("images") if p.is_dir()]
    anns=[p for p in root.rglob("annotations") if p.is_dir()]
    if not imgs or not anns: raise RuntimeError(f"Could not find images/annotations under {root}")
    return imgs[0],anns[0]

DIRS={}
for s,r in roots.items():
    i,a=find_dirs(r); DIRS[s]={"images":i,"annotations":a}
    print(s,"=>",i,a)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""): h.update(c)
    return h.hexdigest()

def group_key(name):
    st=Path(name).stem
    if "_d_" in st: return st.split("_d_")[0]
    q=st.split("_"); return "_".join(q[:2]) if len(q)>=2 else q[0]

def parse_ann(p):
    rec=[]; bad=[]
    if not p.exists(): return rec,[("missing_annotation_file",None,"")]
    for n,line in enumerate(p.read_text(encoding="utf-8-sig",errors="replace").splitlines(),1):
        raw=line.strip()
        if not raw: continue
        v=[x.strip() for x in raw.split(",")]
        if len(v)<8: bad.append(("field_count",n,raw)); continue
        try:
            x,y,w,h=map(float,v[:4]); score=int(float(v[4])); cat=int(float(v[5]))
            tr=int(float(v[6])); oc=int(float(v[7]))
            rec.append((x,y,w,h,score,cat,tr,oc))
        except: bad.append(("parse_error",n,raw))
    return rec,bad

def clip_box(x,y,w,h,W,H):
    x1=max(0.,x); y1=max(0.,y); x2=min(float(W),x+w); y2=min(float(H),y+h)
    return x1,y1,x2-x1,y2-y1

def hbin(h):
    return "<16 px" if h<16 else "16-31 px" if h<32 else "32-63 px" if h<64 else "64-127 px" if h<128 else ">=128 px"

In [ ]:
# CELL 5 — Full raw VisDrone audit
image_rows=[]; box_rows=[]; qc=[]; malformed=[]

for split,d in DIRS.items():
    paths=sorted(p for p in d["images"].iterdir() if p.suffix.lower() in IMAGE_EXTS)
    for ip in tqdm(paths,desc=f"Audit {split}"):
        ap=d["annotations"]/(ip.stem+".txt")
        img=cv2.imread(str(ip))
        if img is None:
            image_rows.append(dict(split=split,filename=ip.name,image_path=str(ip),readable=False,sha256=sha256_file(ip)))
            qc.append(dict(split=split,filename=ip.name,severity="critical",issue="unreadable_image"))
            continue
        H,W=img.shape[:2]; gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
        try: ph=str(imagehash.phash(Image.open(ip).convert("RGB")))
        except: ph=None
        rec,bad=parse_ann(ap)
        for issue,n,raw in bad:
            malformed.append(dict(split=split,filename=ip.name,issue=issue,line=n,raw=raw))
        pc=0
        for x,y,w,h,score,cat,tr,oc in rec:
            raw_ok=(w>0 and h>0 and x>=0 and y>=0 and x+w<=W+1e-6 and y+h<=H+1e-6)
            if not raw_ok: qc.append(dict(split=split,filename=ip.name,severity="warning",issue="raw_box_invalid_or_oob"))
            if cat not in CATS: qc.append(dict(split=split,filename=ip.name,severity="critical",issue=f"unknown_category_{cat}"))
            if score not in (0,1): qc.append(dict(split=split,filename=ip.name,severity="warning",issue=f"unexpected_score_{score}"))
            if tr not in (0,1): qc.append(dict(split=split,filename=ip.name,severity="warning",issue=f"unexpected_truncation_{tr}"))
            if oc not in (0,1,2): qc.append(dict(split=split,filename=ip.name,severity="warning",issue=f"unexpected_occlusion_{oc}"))
            cx,cy,cw,ch=clip_box(x,y,w,h,W,H); valid=cw>0 and ch>0
            person=(cat in PERSON_CATEGORIES and score==1 and valid)
            if person: pc+=1
            area=cw*ch if valid else np.nan
            box_rows.append(dict(
                split=split,filename=ip.name,group_key=group_key(ip.name),image_width=W,image_height=H,
                category_id=cat,category_name=CATS.get(cat,"unknown"),score=score,truncation=tr,occlusion=oc,
                x=cx,y=cy,w=cw,h=ch,canonical_valid=valid,is_person=person,area_px=area,
                area_fraction=area/(W*H) if valid else np.nan,width_fraction=cw/W if valid else np.nan,
                height_fraction=ch/H if valid else np.nan,aspect_ratio=cw/ch if valid and ch>0 else np.nan,
                center_x_norm=(cx+cw/2)/W if valid else np.nan,center_y_norm=(cy+ch/2)/H if valid else np.nan,
                aerial_height_bin=hbin(ch) if valid else None
            ))
        image_rows.append(dict(
            split=split,filename=ip.name,image_path=str(ip),annotation_path=str(ap),group_key=group_key(ip.name),
            readable=True,width=W,height=H,file_size_bytes=ip.stat().st_size,sha256=sha256_file(ip),phash=ph,
            brightness_mean=float(gray.mean()),contrast_std=float(gray.std()),
            blur_laplacian_var=float(cv2.Laplacian(gray,cv2.CV_64F).var()),
            near_black_fraction=float((gray<8).mean()),all_annotation_count=len(rec),
            person_count=pc,person_background=(pc==0)
        ))

images_df=pd.DataFrame(image_rows)
boxes_df=pd.DataFrame(box_rows)
person_df=boxes_df[boxes_df.is_person==True].copy()
qc_df=pd.DataFrame(qc); malformed_df=pd.DataFrame(malformed)

for name,df in {
    "visdrone_image_inventory.csv":images_df,"visdrone_all_boxes.csv":boxes_df,
    "visdrone_person_boxes.csv":person_df,"visdrone_qc_issues.csv":qc_df,
    "visdrone_malformed_annotations.csv":malformed_df}.items():
    df.to_csv(TABLE_DIR/name,index=False)

split_summary=images_df.groupby("split").agg(
    images=("filename","count"),person_boxes=("person_count","sum"),
    background_images=("person_background","sum"),mean_persons=("person_count","mean"),
    median_persons=("person_count","median"),mean_brightness=("brightness_mean","mean"),
    mean_blur=("blur_laplacian_var","mean")).reset_index()
split_summary["background_ratio"]=split_summary.background_images/split_summary.images
split_summary.to_csv(TABLE_DIR/"split_summary.csv",index=False)
display(split_summary)

In [ ]:
# CELL 6 — Exact/near duplicate leakage + distribution-shift tests
from scipy.stats import ks_2samp, chi2_contingency

# Exact duplicates
dup=[]
for sha,g in images_df.groupby("sha256"):
    if len(g)>1:
        ss=sorted(g.split.unique())
        dup.append(dict(sha256=sha,count=len(g),splits="|".join(ss),cross_split=len(ss)>1,files="|".join(g.filename)))
exact_df=pd.DataFrame(dup)
exact_df.to_csv(TABLE_DIR/"exact_duplicates.csv",index=False)

# Group-key overlap
gx=images_df.groupby("group_key").split.nunique()
go=images_df[images_df.group_key.isin(gx[gx>1].index)][["group_key","split","filename"]]
go.to_csv(TABLE_DIR/"group_key_cross_split_overlap.csv",index=False)

# BK-tree perceptual-hash cross-split search
def hd(a,b): return (a^b).bit_count()
class BK:
    def __init__(self): self.root=None
    def add(self,item):
        if self.root is None: self.root=[item,{}]; return
        n=self.root
        while True:
            d=hd(item[0],n[0][0])
            if d in n[1]: n=n[1][d]
            else: n[1][d]=[item,{}]; return
    def query(self,v,t):
        if self.root is None:return []
        out=[]; st=[self.root]
        while st:
            n=st.pop(); d=hd(v,n[0][0])
            if d<=t: out.append((d,n[0]))
            for e,ch in n[1].items():
                if d-t<=e<=d+t: st.append(ch)
        return out

order=[s for s in ["train","val","test-dev"] if s in images_df.split.unique()]
pairs=[]
for i in range(len(order)):
    for j in range(i+1,len(order)):
        a=images_df[(images_df.split==order[i])&images_df.phash.notna()]
        b=images_df[(images_df.split==order[j])&images_df.phash.notna()]
        tree=BK()
        for r in a.itertuples(): tree.add((int(r.phash,16),r.filename,r.sha256))
        for r in tqdm(b.itertuples(),total=len(b),desc=f"pHash {order[i]} vs {order[j]}"):
            for d,item in tree.query(int(r.phash,16),NEAR_DUP_HAMMING_THRESHOLD):
                _,fn,sha=item
                if sha!=r.sha256:
                    pairs.append(dict(split_a=order[i],file_a=fn,split_b=order[j],file_b=r.filename,phash_hamming=d))
                    if len(pairs)>=NEAR_DUP_MAX_PAIRS: break
            if len(pairs)>=NEAR_DUP_MAX_PAIRS: break
near_df=pd.DataFrame(pairs)
near_df.to_csv(TABLE_DIR/"cross_split_near_duplicates.csv",index=False)

# Train-vs-val statistical shift tests
tests=[]
def KS(metric,a,b):
    a=pd.Series(a).replace([np.inf,-np.inf],np.nan).dropna()
    b=pd.Series(b).replace([np.inf,-np.inf],np.nan).dropna()
    if len(a)>1 and len(b)>1:
        r=ks_2samp(a,b); tests.append(dict(test="KS",metric=metric,statistic=r.statistic,p_value=r.pvalue))
if {"train","val"}.issubset(set(images_df.split)):
    KS("persons_per_image",images_df[images_df.split=="train"].person_count,images_df[images_df.split=="val"].person_count)
    KS("brightness",images_df[images_df.split=="train"].brightness_mean,images_df[images_df.split=="val"].brightness_mean)
    KS("blur",images_df[images_df.split=="train"].blur_laplacian_var,images_df[images_df.split=="val"].blur_laplacian_var)
    KS("log_bbox_area",np.log1p(person_df[person_df.split=="train"].area_px),np.log1p(person_df[person_df.split=="val"].area_px))
    KS("bbox_height",person_df[person_df.split=="train"].h,person_df[person_df.split=="val"].h)
shift_df=pd.DataFrame(tests)
shift_df.to_csv(TABLE_DIR/"train_val_distribution_shift_tests.csv",index=False)

print("Cross-split exact duplicate groups:", int(exact_df.cross_split.sum()) if len(exact_df) else 0)
print("Reported cross-split near-duplicate pairs:",len(near_df))
display(shift_df)

In [ ]:
# CELL 7 — Generate graphical figures
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize":(10,6),"figure.dpi":130,"savefig.dpi":180,"axes.grid":True,"grid.alpha":.25})

figs=[]
def S(name):
    p=FIG_DIR/name; plt.tight_layout(); plt.savefig(p,bbox_inches="tight"); plt.show(); plt.close(); figs.append(p)

ss=split_summary.set_index("split")
plt.figure(); ss.images.plot.bar(); plt.title("VisDrone Images per Split"); plt.ylabel("Images"); S("01_images_per_split.png")
plt.figure(); ss.person_boxes.plot.bar(); plt.title("Canonical Person Boxes per Split"); plt.ylabel("Boxes"); S("02_person_boxes_per_split.png")

plt.figure()
for s in order: plt.hist(images_df[images_df.split==s].person_count,bins=50,alpha=.45,label=s)
plt.title("Persons per Image"); plt.xlabel("Persons"); plt.ylabel("Images"); plt.legend(); S("03_persons_per_image.png")

plt.figure()
for s in order: plt.hist(np.log10(person_df[person_df.split==s].area_px.clip(lower=1)),bins=60,alpha=.45,label=s)
plt.title("Person Box Area"); plt.xlabel("log10(area px²)"); plt.ylabel("Boxes"); plt.legend(); S("04_bbox_area.png")

plt.figure()
for s in order: plt.hist(person_df[person_df.split==s].h.clip(upper=300),bins=60,alpha=.45,label=s)
plt.title("Person Box Height"); plt.xlabel("Height px (clipped 300)"); plt.ylabel("Boxes"); plt.legend(); S("05_bbox_height.png")

ho=["<16 px","16-31 px","32-63 px","64-127 px",">=128 px"]
hc=person_df.groupby(["split","aerial_height_bin"]).size().unstack(fill_value=0).reindex(columns=ho,fill_value=0)
plt.figure(figsize=(11,6)); hc.T.plot.bar(ax=plt.gca()); plt.title("Aerial Target Height Bins"); plt.ylabel("Boxes"); S("06_height_bins.png")

oc=person_df.groupby(["split","occlusion"]).size().unstack(fill_value=0)
plt.figure(); oc.T.plot.bar(ax=plt.gca()); plt.title("Occlusion Distribution"); plt.ylabel("Boxes"); S("07_occlusion.png")

tr=person_df.groupby(["split","truncation"]).size().unstack(fill_value=0)
plt.figure(); tr.T.plot.bar(ax=plt.gca()); plt.title("Truncation Distribution"); plt.ylabel("Boxes"); S("08_truncation.png")

plt.figure(figsize=(8,7)); plt.hist2d(person_df.center_x_norm,person_df.center_y_norm,bins=40); plt.colorbar(label="Boxes")
plt.gca().invert_yaxis(); plt.title("Normalized Person Center Density"); plt.xlabel("x"); plt.ylabel("y"); S("09_center_heatmap.png")

plt.figure(); plt.hist(person_df.aspect_ratio.replace([np.inf,-np.inf],np.nan).dropna().clip(upper=3),bins=60)
plt.title("Person Box Aspect Ratio"); plt.xlabel("w/h"); plt.ylabel("Boxes"); S("10_aspect_ratio.png")

plt.figure()
for s in order:
    d=images_df[images_df.split==s]; plt.scatter(d.width,d.height,s=10,alpha=.4,label=s)
plt.title("Image Resolution Distribution"); plt.xlabel("Width"); plt.ylabel("Height"); plt.legend(); S("11_resolution.png")

plt.figure()
for s in order:
    d=images_df[images_df.split==s]; plt.scatter(d.brightness_mean,np.log1p(d.blur_laplacian_var),s=10,alpha=.4,label=s)
plt.title("Brightness vs Sharpness"); plt.xlabel("Mean brightness"); plt.ylabel("log(1+Laplacian variance)"); plt.legend(); S("12_brightness_blur.png")

cc=boxes_df[boxes_df.score==1].category_name.value_counts().sort_values()
plt.figure(figsize=(11,6)); cc.plot.barh(); plt.title("Original VisDrone Class Distribution"); plt.xlabel("Boxes"); S("13_original_classes.png")

bg=ss[["images","background_images"]].copy(); bg["person_images"]=bg.images-bg.background_images
plt.figure(); bg[["person_images","background_images"]].plot.bar(stacked=True,ax=plt.gca())
plt.title("Person-Positive vs Background Images"); plt.ylabel("Images"); S("14_background_ratio.png")

top=images_df.nlargest(20,"person_count")[["filename","split","person_count"]].copy()
top["label"]=top.split+" | "+top.filename
plt.figure(figsize=(11,8)); top.sort_values("person_count").plot.barh(x="label",y="person_count",legend=False,ax=plt.gca())
plt.title("Top 20 Crowded Images"); plt.xlabel("Persons"); S("15_crowded.png")

plt.figure()
for s in order:
    v=np.sort(person_df[person_df.split==s].h.dropna())
    if len(v): plt.plot(v,np.arange(1,len(v)+1)/len(v),label=s)
plt.xscale("log"); plt.title("CDF of Person Box Height"); plt.xlabel("Height px (log)"); plt.ylabel("CDF"); plt.legend(); S("16_height_cdf.png")

print("Figures saved:",len(figs))

In [ ]:
# CELL 8 — Annotated sample contact sheets
def annotated(ip,rows):
    im=Image.open(ip).convert("RGB"); d=ImageDraw.Draw(im)
    for b in rows:
        x,y,w,h=b["x"],b["y"],b["w"],b["h"]; d.rectangle([x,y,x+w,y+h],width=2)
    return im

def sheet(items,cols=4,tw=400,th=280,title=""):
    rows=math.ceil(len(items)/cols); top=40
    can=Image.new("RGB",(cols*tw,rows*th+top),"white"); dr=ImageDraw.Draw(can); dr.text((10,10),title,fill="black")
    for k,(im,cap) in enumerate(items):
        im=im.copy(); im.thumbnail((tw,th-24)); x=(k%cols)*tw; y=(k//cols)*th+top
        can.paste(im,(x,y)); dr.text((x+4,y+th-22),cap[:58],fill="black")
    return can

for s in order:
    d=images_df[(images_df.split==s)&(images_df.readable==True)].sort_values("person_count")
    if d.empty: continue
    ids=sorted(set([0,len(d)//4,len(d)//2,3*len(d)//4,len(d)-1]))
    items=[]
    for i in ids:
        r=d.iloc[i]
        rr=person_df[(person_df.split==s)&(person_df.filename==r.filename)].to_dict("records")
        items.append((annotated(Path(r.image_path),rr),f"{r.filename} | persons={r.person_count}"))
    sh=sheet(items,title=f"VisDrone {s} — Person Samples")
    dst=SAMPLE_DIR/f"annotated_samples_{s}.jpg"; sh.save(dst,quality=92); display(sh)

In [ ]:
# CELL 9 — Canonical person-only YOLO / COCO / VisDrone export
if BUILD_ROOT.exists(): shutil.rmtree(BUILD_ROOT)
BUILD_ROOT.mkdir(parents=True)
YOLO=BUILD_ROOT/"yolo"; COCO=BUILD_ROOT/"coco"; VIS=BUILD_ROOT/"visdrone_person"
for p in [YOLO,COCO,VIS]: p.mkdir(parents=True)

export_splits=[s for s in ["train","val"] if s in DIRS]
manifest=[]; coco_by={}

for s in export_splits:
    yi=YOLO/"images"/s; yl=YOLO/"labels"/s; vi=VIS/"images"/s; va=VIS/"annotations"/s
    for p in [yi,yl,vi,va]: p.mkdir(parents=True,exist_ok=True)
    ims=images_df[images_df.split==s]
    cims=[]; cans=[]; aid=1
    for iid,r in enumerate(tqdm(ims.itertuples(),total=len(ims),desc=f"Export {s}"),1):
        if not r.readable: continue
        src=Path(r.image_path); W,H=int(r.width),int(r.height)
        rr=person_df[(person_df.split==s)&(person_df.filename==r.filename)]
        if EXPORT_IMAGES:
            shutil.copy2(src,yi/r.filename); shutil.copy2(src,vi/r.filename)
        ylines=[]; vlines=[]
        for b in rr.itertuples():
            xc=(b.x+b.w/2)/W; yc=(b.y+b.h/2)/H; nw=b.w/W; nh=b.h/H
            ylines.append(f"0 {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}")
            vlines.append(f"{b.x:.3f},{b.y:.3f},{b.w:.3f},{b.h:.3f},1,1,{int(b.truncation)},{int(b.occlusion)}")
            cans.append({"id":aid,"image_id":iid,"category_id":1,"bbox":[b.x,b.y,b.w,b.h],
                         "area":b.w*b.h,"iscrowd":0,
                         "attributes":{"source_category":b.category_name,"truncation":int(b.truncation),"occlusion":int(b.occlusion)}})
            aid+=1
        (yl/(Path(r.filename).stem+".txt")).write_text("\n".join(ylines)+("\n" if ylines else ""))
        (va/(Path(r.filename).stem+".txt")).write_text("\n".join(vlines)+("\n" if vlines else ""))
        cims.append({"id":iid,"file_name":r.filename,"width":W,"height":H})
        manifest.append({"dataset":"VisDrone2019-DET","split":s,"filename":r.filename,"group_key":r.group_key,
                         "width":W,"height":H,"person_count":int(r.person_count),"background":bool(r.person_background),
                         "source_sha256":r.sha256})
    coco={"info":{"description":"VisDrone person-only: pedestrian + people -> person","version":"step2-v1"},
          "images":cims,"annotations":cans,"categories":[{"id":1,"name":"person","supercategory":"person"}],"licenses":[]}
    (COCO/f"instances_{s}.json").write_text(json.dumps(coco,indent=2)); coco_by[s]=coco

(YOLO/"data.yaml").write_text("path: .\ntrain: images/train\nval: images/val\nnames:\n  0: person\n")
manifest_df=pd.DataFrame(manifest); manifest_df.to_csv(BUILD_ROOT/"dataset_manifest.csv",index=False)
display(manifest_df.groupby("split").agg(images=("filename","count"),persons=("person_count","sum")))

In [ ]:
# CELL 10 — YOLO validation, COCO parity, source registry, checksums and gates
issues=[]; parity=[]
for s in export_splits:
    imdir=YOLO/"images"/s; ldir=YOLO/"labels"/s
    cmap=Counter(a["image_id"] for a in coco_by[s]["annotations"])
    iid={x["file_name"]:x["id"] for x in coco_by[s]["images"]}
    for ip in sorted(p for p in imdir.iterdir() if p.suffix.lower() in IMAGE_EXTS):
        lp=ldir/(ip.stem+".txt")
        if not lp.exists():
            issues.append(dict(split=s,filename=ip.name,severity="critical",issue="missing_label")); continue
        n=0
        for ln,raw in enumerate(lp.read_text().splitlines(),1):
            if not raw.strip(): continue
            q=raw.split()
            if len(q)!=5:
                issues.append(dict(split=s,filename=ip.name,severity="critical",issue=f"field_count_{ln}")); continue
            try: cl,xc,yc,w,h=map(float,q)
            except:
                issues.append(dict(split=s,filename=ip.name,severity="critical",issue=f"parse_{ln}")); continue
            if int(cl)!=0: issues.append(dict(split=s,filename=ip.name,severity="critical",issue=f"class_{cl}"))
            if not(0<=xc<=1 and 0<=yc<=1 and 0<w<=1 and 0<h<=1):
                issues.append(dict(split=s,filename=ip.name,severity="critical",issue=f"normalized_box_{ln}"))
            if xc-w/2<-1e-6 or xc+w/2>1+1e-6 or yc-h/2<-1e-6 or yc+h/2>1+1e-6:
                issues.append(dict(split=s,filename=ip.name,severity="critical",issue=f"oob_{ln}"))
            n+=1
        parity.append(dict(split=s,filename=ip.name,yolo_box_count=n,coco_box_count=cmap.get(iid[ip.name],0),
                           match=n==cmap.get(iid[ip.name],0)))

yolo_issues=pd.DataFrame(issues); parity_df=pd.DataFrame(parity)
yolo_issues.to_csv(TABLE_DIR/"canonical_yolo_validation_issues.csv",index=False)
parity_df.to_csv(TABLE_DIR/"yolo_coco_parity.csv",index=False)

source=pd.DataFrame([{"dataset":"VisDrone2019-DET","role":"primary development dataset",
 "source":"AISKYEYE / Tianjin University","official_repository":"https://github.com/VisDrone/VisDrone-Dataset",
 "person_mapping":"pedestrian(1) + people(2) -> person",
 "notes":"Official DET benchmark; test-dev is reference only, not the private final test."}])
source.to_csv(PROV_DIR/"source_registry.csv",index=False)

cks=[]
for p in list(archives.values())+[TABLE_DIR/"visdrone_image_inventory.csv",TABLE_DIR/"visdrone_person_boxes.csv",BUILD_ROOT/"dataset_manifest.csv"]:
    cks.append({"file":str(p),"sha256":sha256_file(p),"bytes":p.stat().st_size})
pd.DataFrame(cks).to_csv(PROV_DIR/"checksums.csv",index=False)

critical=int((qc_df.severity=="critical").sum()) if len(qc_df) else 0
cross_exact=int(exact_df.cross_split.sum()) if len(exact_df) else 0
ycrit=int((yolo_issues.severity=="critical").sum()) if len(yolo_issues) else 0
pmis=int((~parity_df.match).sum()) if len(parity_df) else 0

gate=pd.DataFrame([
 {"gate":"Raw images readable","value":int(images_df.readable.sum()),"status":"PASS" if images_df.readable.all() else "FAIL"},
 {"gate":"Malformed annotation rows","value":len(malformed_df),"status":"PASS" if len(malformed_df)==0 else "FAIL"},
 {"gate":"Critical raw QC issues","value":critical,"status":"PASS" if critical==0 else "FAIL"},
 {"gate":"Cross-split exact duplicate groups","value":cross_exact,"status":"PASS" if cross_exact==0 else "FAIL"},
 {"gate":"Cross-split near-duplicate pairs","value":len(near_df),"status":"PASS" if len(near_df)==0 else "WARN"},
 {"gate":"Canonical YOLO critical issues","value":ycrit,"status":"PASS" if ycrit==0 else "FAIL"},
 {"gate":"YOLO/COCO parity mismatches","value":pmis,"status":"PASS" if pmis==0 else "FAIL"},
 {"gate":"Private final test >=1000 images","value":"not yet audited","status":"PENDING"}])
gate.to_csv(REPORT_DIR/"step2_gate_report.csv",index=False)
display(gate)

In [ ]:
# CELL 11 — Automated summary + HTML/PDF report
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate,Paragraph,Spacer,Table,TableStyle,Image as RI,PageBreak

summary={
 "dataset":"VisDrone2019-DET","task":"person-only aerial object detection",
 "mapping":{"pedestrian":"person","people":"person"},"splits_analyzed":order,
 "images_total":int(len(images_df)),"person_boxes_total":int(len(person_df)),
 "background_images_total":int(images_df.person_background.sum()),
 "exact_duplicate_groups":int(len(exact_df)),"cross_split_exact_duplicate_groups":cross_exact,
 "cross_split_near_duplicate_pairs":int(len(near_df)),"malformed_rows":int(len(malformed_df)),
 "canonical_yolo_critical_issues":ycrit,"yolo_coco_parity_mismatches":pmis,
 "private_final_test":{"minimum_images":1000,"status":"PENDING"}}
(REPORT_DIR/"step2_summary.json").write_text(json.dumps(summary,indent=2))

# HTML
html=["<html><head><meta charset='utf-8'><style>body{font-family:Arial;max-width:1200px;margin:auto;padding:24px}table{border-collapse:collapse;width:100%}th,td{border:1px solid #ddd;padding:6px}img{max-width:100%}</style></head><body>",
"<h1>Step 2 — VisDrone Person-Only Automated Audit</h1><h2>Split summary</h2>",
split_summary.to_html(index=False),"<h2>Gate report</h2>",gate.to_html(index=False),"<h2>Shift tests</h2>",
shift_df.to_html(index=False) if len(shift_df) else "<p>None</p>","<h2>Figures</h2>"]
for p in figs:
    b=base64.b64encode(p.read_bytes()).decode()
    html.append(f"<h3>{p.stem}</h3><img src='data:image/png;base64,{b}'>")
html.append("</body></html>")
(REPORT_DIR/"Step2_VisDrone_Automated_Audit_Report.html").write_text("\n".join(html),encoding="utf-8")

# PDF
pdf=REPORT_DIR/"Step2_VisDrone_Automated_Audit_Report.pdf"
doc=SimpleDocTemplate(str(pdf),pagesize=A4,rightMargin=1.2*cm,leftMargin=1.2*cm,topMargin=1.2*cm,bottomMargin=1.2*cm)
styles=getSampleStyleSheet(); story=[Paragraph("Step 2 — VisDrone Person-Only Automated Dataset Audit",styles["Title"]),Spacer(1,12)]
for title,df in [("Split summary",split_summary),("Step-2 gate report",gate)]:
    story.append(Paragraph(title,styles["Heading2"]))
    data=[list(df.columns)]+df.round(3).astype(str).values.tolist()
    t=Table(data,repeatRows=1); t.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,0),colors.lightgrey),("GRID",(0,0),(-1,-1),.4,colors.grey),("FONTSIZE",(0,0),(-1,-1),7)]))
    story += [t,Spacer(1,12)]
story.append(PageBreak())
for p in figs:
    story.append(Paragraph(p.stem.replace("_"," "),styles["Heading2"]))
    im=RI(str(p)); r=min(18*cm/im.imageWidth,11.5*cm/im.imageHeight); im.drawWidth*=r; im.drawHeight*=r
    story += [im,Spacer(1,10)]
doc.build(story)
print("Reports saved:",REPORT_DIR)

In [ ]:
# CELL 12 — Create Drive delivery ZIPs
canonical_zip=DELIVERY_DIR/"VisDrone_PersonOnly_Canonical_Step2_v1.zip"
if canonical_zip.exists(): canonical_zip.unlink()
print("Creating canonical dataset ZIP...")
shutil.make_archive(str(canonical_zip.with_suffix("")),"zip",root_dir=BUILD_ROOT)

compact=LOCAL_ROOT/"compact"
if compact.exists(): shutil.rmtree(compact)
compact.mkdir()
shutil.copytree(REPORT_DIR,compact/"reports")
shutil.copytree(PROV_DIR,compact/"provenance")
reports_zip=DELIVERY_DIR/"Step2_VisDrone_Reports_Provenance_v1.zip"
if reports_zip.exists(): reports_zip.unlink()
shutil.make_archive(str(reports_zip.with_suffix("")),"zip",root_dir=compact)

print("Canonical dataset:",canonical_zip)
print("Reports/provenance:",reports_zip)
print("Everything saved under:",DRIVE_ROOT)

# FINAL CELL — Audit the dataset exported by Aerial Person Studio GUI

After the GUI dataset is ready, change only `CUSTOM_GUI_DATASET_ROOT`.  
The cell auto-detects a YOLO export and checks:

- image/label pairing;
- class IDs;
- normalized box validity;
- out-of-bounds boxes;
- background images;
- exact cross-split duplicates;
- person-size distribution;
- **private final test ≥ 1,000 images**.

In [ ]:
# FINAL CELL — Custom GUI dataset audit
CUSTOM_GUI_DATASET_ROOT=Path("/content/drive/MyDrive/AerialHumanDetection/Custom_Aerial_Dataset_From_GUI")
CUSTOM_OUT=DRIVE_ROOT/"05_custom_gui_dataset_audit"; CUSTOM_OUT.mkdir(parents=True,exist_ok=True)

def find_yolo(root):
    for c in [root,root/"outputs"/"yolo",root/"yolo",root/"export"/"yolo",root/"exports"/"yolo"]:
        if (c/"images").exists() and (c/"labels").exists(): return c
    if root.exists():
        for p in root.rglob("images"):
            if (p.parent/"labels").exists(): return p.parent
    return None

def audit_custom(root):
    if not root.exists():
        print("Custom GUI dataset not found. Set CUSTOM_GUI_DATASET_ROOT later; no error.")
        return None
    yr=find_yolo(root)
    if yr is None: raise FileNotFoundError("Could not locate YOLO images/labels under custom root.")
    ims=[]; bxs=[]; errs=[]
    ss=[s for s in ["train","val","test"] if (yr/"images"/s).exists()]
    for s in ss:
        idir=yr/"images"/s; ldir=yr/"labels"/s
        ipaths=sorted(p for p in idir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
        lmap={p.stem:p for p in ldir.glob("*.txt")} if ldir.exists() else {}
        for ip in tqdm(ipaths,desc=f"Custom {s}"):
            im=cv2.imread(str(ip))
            if im is None:
                errs.append(dict(split=s,filename=ip.name,severity="critical",issue="unreadable")); continue
            H,W=im.shape[:2]; lp=lmap.get(ip.stem); n=0
            if lp is None:
                errs.append(dict(split=s,filename=ip.name,severity="critical",issue="missing_label")); lines=[]
            else: lines=[x.strip() for x in lp.read_text().splitlines() if x.strip()]
            for k,raw in enumerate(lines,1):
                q=raw.split()
                if len(q)!=5: errs.append(dict(split=s,filename=ip.name,severity="critical",issue=f"field_count_{k}")); continue
                try: cl,xc,yc,nw,nh=map(float,q)
                except: errs.append(dict(split=s,filename=ip.name,severity="critical",issue=f"parse_{k}")); continue
                if int(cl)!=0: errs.append(dict(split=s,filename=ip.name,severity="critical",issue=f"class_{cl}"))
                if not(0<=xc<=1 and 0<=yc<=1 and 0<nw<=1 and 0<nh<=1):
                    errs.append(dict(split=s,filename=ip.name,severity="critical",issue=f"normalized_{k}")); continue
                if xc-nw/2<-1e-6 or xc+nw/2>1+1e-6 or yc-nh/2<-1e-6 or yc+nh/2>1+1e-6:
                    errs.append(dict(split=s,filename=ip.name,severity="critical",issue=f"oob_{k}")); continue
                bh=nh*H; bw=nw*W; n+=1
                bxs.append(dict(split=s,filename=ip.name,w_px=bw,h_px=bh,area_px=bw*bh,
                                aspect_ratio=bw/bh if bh else np.nan,aerial_height_bin=hbin(bh)))
            gray=cv2.cvtColor(im,cv2.COLOR_BGR2GRAY)
            ims.append(dict(split=s,filename=ip.name,width=W,height=H,person_count=n,background=n==0,
                            brightness_mean=float(gray.mean()),blur_laplacian_var=float(cv2.Laplacian(gray,cv2.CV_64F).var()),
                            sha256=sha256_file(ip)))
    I=pd.DataFrame(ims); B=pd.DataFrame(bxs); E=pd.DataFrame(errs)
    dups=[]
    for sha,g in I.groupby("sha256"):
        if len(g)>1 and g.split.nunique()>1: dups.append(dict(sha256=sha,splits="|".join(sorted(g.split.unique())),files="|".join(g.filename)))
    D=pd.DataFrame(dups)
    I.to_csv(CUSTOM_OUT/"custom_image_inventory.csv",index=False); B.to_csv(CUSTOM_OUT/"custom_person_boxes.csv",index=False)
    E.to_csv(CUSTOM_OUT/"custom_qc_issues.csv",index=False); D.to_csv(CUSTOM_OUT/"custom_cross_split_exact_duplicates.csv",index=False)
    sm=I.groupby("split").agg(images=("filename","count"),boxes=("person_count","sum"),background_images=("background","sum"),mean_persons=("person_count","mean")).reset_index()
    sm.to_csv(CUSTOM_OUT/"custom_split_summary.csv",index=False)
    crit=int((E.severity=="critical").sum()) if len(E) else 0
    testn=int((I.split=="test").sum()) if "test" in set(I.split) else 0
    G=pd.DataFrame([
        {"gate":"Critical custom QC issues","value":crit,"status":"PASS" if crit==0 else "FAIL"},
        {"gate":"Cross-split exact duplicate groups","value":len(D),"status":"PASS" if len(D)==0 else "FAIL"},
        {"gate":"Private final test >=1000 images","value":testn,"status":"PASS" if testn>=1000 else "FAIL"}])
    G.to_csv(CUSTOM_OUT/"custom_step2_gate_report.csv",index=False)
    display(sm); display(G)
    plt.figure(figsize=(8,5)); sm.set_index("split").images.plot.bar(); plt.title("Custom Dataset Images per Split"); plt.ylabel("Images")
    plt.tight_layout(); plt.savefig(CUSTOM_OUT/"custom_images_per_split.png",dpi=180); plt.show(); plt.close()
    if len(B):
        plt.figure(figsize=(9,5))
        for s in sm.split:
            v=B[B.split==s].h_px
            if len(v): plt.hist(v.clip(upper=300),bins=50,alpha=.45,label=s)
        plt.title("Custom Person Box Height"); plt.xlabel("Height px"); plt.ylabel("Boxes"); plt.legend()
        plt.tight_layout(); plt.savefig(CUSTOM_OUT/"custom_person_height_distribution.png",dpi=180); plt.show(); plt.close()
    return {"images":I,"boxes":B,"issues":E,"duplicates":D,"summary":sm,"gate":G}

custom_audit=audit_custom(CUSTOM_GUI_DATASET_ROOT)